In [12]:
import os
import pandas as pd
import random
import numpy as np
from scipy.stats import hypergeom
from goatools import obo_parser
from goatools.associations import read_gaf
file_name= 'genomic.gtf'
# get project path

path = os.path.join(os.path.dirname(os.getcwd()),'metadata',file_name)
df = pd.read_csv(path, sep='\t', comment='#', header=None)
df.columns = ['seqid', 'source', 'feature', 'start', 'end', 'score', 'strand', 'phase', 'attributes']
df = df[df['feature'] == 'CDS']
# create dictionary to store gene id and gene name
gene_id_name = {}
for index, row in df.iterrows():
    gene_id = row['attributes'].split(';')[2].split(' ')[2].replace('"', '')
    #remove 'GeneID:' from gene_id
    gene_id = gene_id.split(':')[-1]
    gene_name = row['attributes'].split(';')[4].split(' ')[2].replace('"', '')
    gene_id_name[gene_name.lower()] = gene_id

def get_lfc_thresh(deseq_df, target_size, fold, p_val_thresh=0.01):
    """
    Function to get the log fold change threshold for a given p-value threshold and target number of DEGs
    :param deseq_df: DataFrame containing DESeq2 results
    :param target_size: target number of DEGs
    :param p_val_thresh: p-value threshold
    :return: log fold change threshold
    """
    df_sorted = deseq_df.copy()
    # filter out genes that do not pass the p-value threshold
    df_sorted = df_sorted[df_sorted['padj'] < p_val_thresh]
    # Sort the DESeq2 results by LFC
    df_sorted = df_sorted.sort_values('log2FoldChange', ascending=False)
    # Get the LFC threshold for the target number of DEGs
    if fold == 'up':
        lfc_thresh = np.maximum(df_sorted.iloc[target_size]['log2FoldChange'],1)
    elif fold == 'down':
        lfc_thresh = np.minimum(df_sorted.iloc[-target_size]['log2FoldChange'],-1)
    else:
        raise ValueError("fold must be 'up' or 'down'")
    return lfc_thresh


def remove_unidentified_genes(deseq_df):
    """
    Function to remove genes that are not identified
    :param df: DataFrame containing the gene names
    :return: DataFrame with the unidentified genes removed
    """
    # Remove genes that are not identified
    ind = [val.lower() in gene_id_name.keys() for val in deseq_df.index.values]
    df_filtered = deseq_df.iloc[ind]
    return df_filtered


def get_deg_genes(path_to_deseq, target_size, fold, p_cutoff):
    """
    Function to get the differentially expressed genes from the DESeq2 results
    :param deseq_df: DataFrame containing DESeq2 results
    :param p_val_thresh: p-value threshold
    :param fold: 'up' or 'down'
    :return: set of differentially expressed genes
    """
    deg_results = pd.read_csv(path_to_deseq, index_col=0)
    deg_results = remove_unidentified_genes(deg_results)
    # Read DEG and background gene lists
    # get the log fold change threshold
    lfc_cutoff = get_lfc_thresh(deg_results, target_size, fold, p_val_thresh=p_cutoff)
    if fold == 'up':
        deg_genes = set(deg_results.index[(deg_results['padj'] < p_cutoff) & (deg_results['log2FoldChange']> lfc_cutoff)])
    elif fold == 'down':
        deg_genes = set(deg_results.index[(deg_results['padj'] < p_cutoff) & (deg_results['log2FoldChange']< lfc_cutoff)])
    
    background_genes = set(deg_results.index)
    return deg_genes, background_genes


def gene_name_to_id(gene_list):
    gene_IDs = []
    for gene in gene_list:
        try:
            gene_IDs.append(gene_id_name[gene.lower()])
        except:
            pass    
    return gene_IDs


def enrichment_score(gene_set, go_term_genes, background_genes):
    #convert to sets to avoid duplicates
    gene_set = set(gene_set)
    go_term_genes = set(go_term_genes)
    background_genes = set(background_genes)
    # Calculate hypergeometric enrichment score
    N = len(background_genes)
    K = len(go_term_genes & background_genes)  # Total genes annotated to the GO term
    n = len(gene_set)
    x = len(gene_set & go_term_genes)
    # Return -log10(p-value) as enrichment strength (larger = more enriched)
    pval = hypergeom.sf(x - 1, N, K, n)
    return -np.log10(pval + 1e-300)  # Avoid log(0)


def go_term_diff_permutation_test(genes_A, genes_B, go_term_genes, background_genes, n_permutations=1000):
    n_A = len(genes_A)
    n_B = len(genes_B)

    # observed enrichment difference
    score_A = enrichment_score(genes_A, go_term_genes, background_genes)
    score_B = enrichment_score(genes_B, go_term_genes, background_genes)
    observed_diff = abs(score_A - score_B)

    # permutation
    diffs = []
    for _ in range(n_permutations):
        perm_A = random.sample(background_genes, n_A)
        perm_B = random.sample(background_genes, n_B)
        diff = abs(enrichment_score(perm_A, go_term_genes, background_genes) -
                   enrichment_score(perm_B, go_term_genes, background_genes))
        diffs.append(diff)

    # empirical p-value
    more_extreme = sum(1 for d in diffs if d >= observed_diff)
    p_empirical = more_extreme / n_permutations

    return {
        "observed_diff": observed_diff,
        "p_empirical": p_empirical,
        "score_A": score_A,
        "score_B": score_B
    }


def get_GO_gene_list(go_term):
    # path to metadata files
    dir = os.path.join(os.path.dirname(os.getcwd()), 'metadata')
    # Define file paths
    GO_OBO = "go-basic.obo"
    GAF_FILE = "ecocyc.gaf"  # Replace with your GAF file
    # Load GO DAG (ontology structure)
    go_dag = obo_parser.GODag(os.path.join(dir, GO_OBO))

    # Load gene-GO associations (you can download GAF files from Gene Ontology site)
    assoc = read_gaf(os.path.join(dir, GAF_FILE))

    # Get genes annotated to the term or its children
    def get_genes_for_term(term, associations, go_dag):
        terms = {t for t in go_dag[term].get_all_children()}
        terms.add(term)
        genes = {gene for gene, go_terms in associations.items() if go_terms & terms}
        return genes

    genes = get_genes_for_term(go_term, assoc, go_dag)
    return list(genes)

In [11]:
dir = os.path.join(os.path.dirname(os.getcwd()),'metadata')
# Define file paths
GO_OBO = "go-basic.obo"
GAF_FILE = "ecocyc.gaf"  # Replace with your GAF file
# define cutoffs
p_cutoff = 0.01
target_size = 1000

# load DEG results
# study sample A:
study_sample = 'regulated'
fold = 'down'
file_name = f'deseq2_results_{study_sample}.csv'
path = os.path.join(os.path.dirname(os.getcwd()),'results','deseq_results','from counts',file_name)
deg_genes_A, background_genes = get_deg_genes(path, target_size, fold, p_cutoff)
# translate to gene ID:
deg_genes_A = gene_name_to_id(deg_genes_A)
background_genes = gene_name_to_id(background_genes)
# sample B
study_sample = 'disrupted'
fold = 'down'
file_name = f'deseq2_results_{study_sample}.csv'
path = os.path.join(os.path.dirname(os.getcwd()),'results','deseq_results','from counts',file_name)
deg_genes_B, _ = get_deg_genes(path, target_size, fold, p_cutoff)
# translate to gene ID:
deg_genes_B = gene_name_to_id(deg_genes_B)


C:\Users\owner\Documents\Projects\rnaseq_correlations\metadata\go-basic.obo: fmt(1.2) rel(2024-09-08) 44,296 Terms
HMS:0:00:00.675941  57,793 annotations READ: C:\Users\owner\Documents\Projects\rnaseq_correlations\metadata\ecocyc.gaf 
3401 IDs in loaded association branch, BP


{'observed_diff': 2.59360263173156,
 'p_empirical': 0.003996003996003996,
 'score_A': 7.321366937612345,
 'score_B': 4.727764305880785}

In [ ]:
# Load the GO enrichment results
file_name = 'GOATOOLS_GO_enrichment_results_disrupted_down.csv'
path = os.path.join(os.path.dirname(os.getcwd()), 'results', 'GO_results', 'from_counts', file_name)
go_results = pd.read_csv(path)
# load the second condition
file_name = 'GOATOOLS_GO_enrichment_results_regulated_down.csv'
path = os.path.join(os.path.dirname(os.getcwd()), 'results', 'GO_results', 'from_counts', file_name)
go_results2 = pd.read_csv(path)
# filter go terms that appear in both conditions
common_go_terms = set(go_results['GO_ID']).intersection(set(go_results2['GO_ID']))
GO_diff_p_val = {}
for go_term in common_go_terms:
   perm_test = go_term_diff_permutation_test(deg_genes_A, deg_genes_B, get_GO_gene_list(go_term), background_genes, n_permutations=10000)
   GO_diff_p_val[go_term] = perm_test['p_empirical']
GO_diff_p_val

In [27]:
# convert to dataframe
s = pd.Series(GO_diff_p_val, name='p-val')
s.values

array([2.653e-01, 1.000e+00, 5.710e-02, 1.000e+00, 0.000e+00, 1.950e-02,
       1.840e-02, 0.000e+00, 7.610e-02, 0.000e+00, 1.000e+00, 1.000e+00,
       1.000e+00, 2.000e-03, 1.000e+00, 1.000e+00, 1.000e+00, 1.000e-04,
       1.000e+00, 9.400e-03])

In [28]:
from statsmodels.stats.multitest import multipletests
# 2. Run BH correction via statsmodels
reject, p_adj, _, _ = multipletests(s.values, method='fdr_bh')

# 3. Build a DataFrame and add the adjusted p-values
df = s.to_frame()            # makes a DataFrame with one column "p_value"
df['p_adj'] = p_adj          # adds a second column "p_adj"

print(df)

             p-val     p_adj
GO:0022904  0.2653  0.482364
GO:0042938  1.0000  1.000000
GO:0009060  0.0571  0.126889
GO:0035351  1.0000  1.000000
GO:0071973  0.0000  0.000000
GO:0044780  0.0195  0.048750
GO:0030254  0.0184  0.048750
GO:0006935  0.0000  0.000000
GO:0015031  0.0761  0.152200
GO:0002181  0.0000  0.000000
GO:0006164  1.0000  1.000000
GO:0015986  1.0000  1.000000
GO:0008360  1.0000  1.000000
GO:0000027  0.0020  0.008000
GO:0006754  1.0000  1.000000
GO:0006189  1.0000  1.000000
GO:0042777  1.0000  1.000000
GO:0044781  0.0001  0.000500
GO:0009252  1.0000  1.000000
GO:0006412  0.0094  0.031333
